<a href="https://colab.research.google.com/github/ArtyomShabunin/SMOPA-26/blob/main/lesson_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://prana-system.com/files/110/rds_color_full.png" alt="tot image" width="300"  align="center"/> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img src="https://mpei.ru/AboutUniverse/OficialInfo/Attributes/PublishingImages/logo1.jpg" alt="mpei image" width="200" align="center"/>
<img src="https://mpei.ru/Structure/Universe/tanpe/structure/tfhe/PublishingImages/tot.png" alt="tot image" width="100"  align="center"/>

---

# **Системы машинного обучения и предиктивной аналитики в тепловой и возобновляемой энергетике**  

# ***Практические занятия***


---

# Занятие №8
# Прогнозирование временных рядов методами машинного обучения
**8 апреля 2025г.**

---

**Временной ряд** — это последовательность числовых данных, упорядоченных во времени и собранных через **равные промежутки времени**.

**Примеры временных рядов:**
- Температура воздуха каждый час
- Цена на нефть каждый день
- Электрическая мощность установки каждую секунду
- Количество клиентов в магазине каждый день
- Давление в котле каждые 10 секунд

---

**Особенности временных рядов:**

1. **Время имеет значение** — порядок данных важен, в отличие от обычных таблиц.
2. **Зависимость от прошлого** — текущее значение может зависеть от предыдущих (автокорреляция).
3. **Стационарность или нестационарность** — поведение ряда может меняться со временем (например, тренд, изменение дисперсии).
4. **Сезонность** — повторяющиеся циклы (день-ночь, зима-лето и т.д.).

---

**Где используются:**

- Финансы (курсы акций, криптовалюты)
- Энергетика (нагрузка, температура, давление)
- Промышленность (сигналы с датчиков)
- Метеорология (погода, климат)
- Медицина (ЭКГ, пульс, мониторинг пациентов)
- Транспорт (потоки, GPS-координаты)

---

**Задача предсказания временных рядов** — это тип задачи машинного обучения или статистического анализа, в которой требуется спрогнозировать будущие значения некоторой переменной на основе её предыдущих наблюдений, упорядоченных во времени.

**Пример:**
Допустим, у нас есть данные о температуре каждый день за последние 30 дней. Мы хотим предсказать температуру на следующий день. Это и есть задача предсказания временного ряда.

**Типы задач:**
- **Одношаговое предсказание (one-step forecasting)** — предсказание значения на следующий момент времени.
- **Многопериодное предсказание (multi-step forecasting)** — предсказание значений на несколько будущих шагов.
- **Унивариантное (univariate)** — используется только один временной ряд.
- **Мультивариантное (multivariate)** — используются несколько временных рядов (например, температура, давление и влажность одновременно).

In [ ]:
# !python -m pip install "pandas<3" "cmdstanpy<1.3"

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Загрузка данных и обработка данных
Набор содержит данные о почасовом производстве ветряной и солнечной электроэнергии (в МВт) во французской электросети с 2020 года.

In [ ]:
import gdown
import warnings
warnings.filterwarnings('ignore')
gdown.download('https://drive.google.com/uc?id=1NAYPaEkovk7jvaURdjI0nCi7CUMxry7W', verify=False)

df = pd.read_csv('./intermittent-renewables-production-france.csv')

In [ ]:
df.head()

In [ ]:
df = df.rename(columns={'Date and Hour' : 'DateTime'})
df['DateTime'] = df['DateTime'].str.slice(stop=-6)
df['DateTime'] = pd.to_datetime(df['DateTime'])
df = df.sort_values(ascending=True,by='DateTime')
df = df.drop(['Date','dayOfYear','dayName','monthName'],axis=1)
df = df.dropna()
df = df.set_index("DateTime")

In [ ]:
df.head()

In [ ]:
solar = df[df['Source'] == 'Solar']['Production']
wind = df[df['Source'] == 'Wind']['Production']

### Диагностика исходных временных рядов



In [ ]:
for name, series in [('Solar', solar), ('Wind', wind)]:
    nan_share = series.isna().mean() * 100
    zero_share = (series == 0).mean() * 100
    step_mode = series.index.to_series().diff().mode().iloc[0]

    print(f"{name}: n={len(series)}, NaN={nan_share:.2f}%, zeros={zero_share:.2f}%")
    print(f"{name}: index_is_unique={series.index.is_unique}, index_sorted={series.index.is_monotonic_increasing}, typical_step={step_mode}")


In [ ]:
# Посомтрим на дубликаты
dup_mask = wind.index.duplicated(keep=False)
dup_idx = wind.index[dup_mask]

print("Число строк с дублирующимся индексом:", dup_mask.sum())
print("Число уникальных дублирующихся timestamp:", dup_idx.nunique())
print("Список timestamp с дублями:")
print(dup_idx.unique().sort_values())

# Посмотреть сами строки с дублями
wind_dup_rows = wind[dup_mask].sort_index()
display(wind_dup_rows)

In [ ]:
# Удаляем дубликаты

solar = solar.groupby(level=0).mean()
wind = wind.groupby(level=0).mean()

In [ ]:
solar, wind

In [ ]:
color_pal = sns.color_palette()
solar.plot(style='.',
          figsize=(20, 5),
          ms=3,
          color=color_pal[3],
          title='Солнечная электроэнергия')
plt.ylabel("МВт")
plt.show()

wind.plot(style='.',
          figsize=(20, 5),
          ms=3,
          color=color_pal[2],
          title='Ветряная электроэнергия')
plt.ylabel("МВт")
plt.show()

## Производство солнечной электроэнергии
### Разложение временного ряда

**Разложение временного ряда** — это метод, при котором временной ряд представляется как сумма (или произведение) нескольких более простых компонент:

1. **Тренд (`trend`)** — общее направление изменения данных с течением времени. Это может быть рост, спад или стабилизация.
2. **Сезонность (`seasonal`)** — периодические колебания, которые повторяются через равные интервалы времени (например, дни недели, месяцы, сезоны года).
3. **Остаток (`residual` или `noise`)** — всё, что не объясняется трендом и сезонностью. Это случайные, непредсказуемые флуктуации.

<img src="https://github.com/ArtyomShabunin/SMOPA-25/blob/main/imgs/trend_seasonality.png?raw=true" alt="trend_seasonality" width="800"  align="center"/>
* картинка из конспекта лекции Воронцова В.К. Методы машинного обучения. Инкрементное и онлайн обучение. ВМК МГУ 2022

- **Ряд 1** - сезонность без тренда
- **Ряд 2** - линейный тренд, аддитивная сезоность
- **Ряд 3** - линейный тренд, мультипликативная сезонность
- **Ряд 4** - экспоненциальный тренд, мультипликативная сезонность

---

**Математические модели**

Есть два основных способа описания этих компонентов:

1. **Аддитивная модель** (additive model)
Предполагает, что все компоненты **независимы друг от друга** и просто **суммируются**:

$$Y_t = T_t + S_t + R_t$$

Где:
- $ Y_t $ — наблюдаемое значение временного ряда в момент времени \( t \)
- $ T_t $ — тренд
- $ S_t $ — сезонная компонента
- $ R_t $ — остаток (шум)

**Когда использовать**: когда амплитуда сезонных колебаний **постоянна**, независимо от уровня тренда.

---

2. **Мультипликативная модель** (multiplicative model)

Предполагает, что компоненты взаимодействуют **мультипликативно**:

$$Y_t = T_t \times S_t \times R_t$$

**Когда использовать**: когда сезонные колебания **усиливаются или ослабевают** вместе с ростом тренда (например, расходы растут и колебания становятся больше по мере роста доходов компании).

---

**Что даёт разложение?**

1. **Анализ структуры ряда** — можно отдельно рассмотреть, какие сезонные эффекты присутствуют, каков общий тренд.
2. **Предобработка для прогнозирования** — если удалить сезонность и тренд, можно подавать чистые остатки на модель.
3. **Детекция аномалий** — если резидуальная компонента аномально большая, можно предположить сбой или событие.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

def seasonal_decompose_plotter(
    df: pd.DataFrame, model='additive',
    period=12, title='', figsize=(20, 12)):

    # period - период сезонности

    decomposition = seasonal_decompose(df.values, model=model, period=period)
    de_season = decomposition.seasonal
    de_resid = decomposition.resid
    de_trend = decomposition.trend

    fig, ax = plt.subplots(4, sharex=True, figsize=figsize)

    ax[0].set_title(title)
    ax[0].plot(df.index, df.values, color='C3')
    ax[0].set_ylabel(df.keys()[0])
    ax[0].grid(alpha=0.25)

    ax[1].plot(df.index, de_trend, color='C1')
    ax[1].set_ylabel('Trend')
    ax[1].grid(alpha=0.25)

    ax[2].plot(df.index, de_season, color='C2')
    ax[2].set_ylabel('Seasonal')
    ax[2].grid(alpha=0.25)

    ax[3].axhline(y=0, color='k', linewidth=1)
    ax[3].scatter(df.index, de_resid, color='C0', s=10)
    ax[3].set_ylabel('Resid')
    ax[3].grid(alpha=0.25)

    plt.tight_layout(h_pad=0)
    plt.show()

    return decomposition

In [ ]:
_ = seasonal_decompose_plotter(
    solar,
    model='additive',
    # model='multiplicative',
    period=24*365,
    title='Solar Power Generation Seasonal Decompose', figsize=(20, 12))

In [ ]:
fig = plt.figure(figsize=(20, 3));
plt.plot(solar.index, _.seasonal, color='C2');
plt.ylabel('Seasonal');
plt.xlim(pd.to_datetime(['2022-12-01', '2022-12-10']));

### Предсказание производства солнечной электроэнергии, на осредненных данных

Для упрощения выполним агрегацию временного ряда с переходом от почасовой к суточной дискретности.

In [ ]:
solar_dy_day = solar.resample('1d').sum()

In [ ]:
solar_dy_day.head()

**Prophet** — это библиотека для **прогнозирования временных рядов**. Она особенно удобна для пользователей, которым **нужны точные прогнозы без глубокого погружения в статистику** или машинное обучение.

---

**Основные особенности Prophet:**

**Удобство** — прост в использовании.  
**Поддерживает тренды и сезонность** — автоматически выявляет и моделирует их.  
**Гибкость** — позволяет добавлять пользовательские праздничные дни, внешние факторы, ручные настройки.  

---

**Как работает Prophet?**

Prophet использует **аддитивную модель**:

$$y(t) = g(t) + s(t) + h(t) + \varepsilon_t$$

где:

- $ g(t) $ — **тренд** (например, линейный или с изменением наклона)
- $ s(t) $ — **сезонность** (дневная, недельная, годовая)
- $ h(t) $ — **праздники или особые события**
- $ \varepsilon_t $ — **ошибка или шум**

Но **сезонность (и праздники)** могут быть как **аддитивными**, так и **мультипликативными** по отношению к тренду.


**Аддитивная сезонность:**

$$y(t) = g(t) + s(t)$$

- Сезонность остаётся одинаковой при любом уровне тренда.
- Подходит, когда амплитуда сезонных колебаний постоянна.

**Мультипликативная сезонность:**
$$
y(t) = g(t) \times (1 + s(t))
$$

- Амплитуда сезонных колебаний **зависит от уровня тренда**.
- Подходит, если сезонные колебания становятся **больше при росте** тренда (например, увеличение спроса и сезонных всплесков одновременно).


Хотя **Prophet** не требует большого количества настройки (он "из коробки" даёт неплохие результаты), **гиперпараметры всё же есть**, и **их тюнинг может существенно улучшить качество прогноза**.

---

**Основные гиперпараметры Prophet, которые можно настраивать:**

| Параметр | Назначение |
|----------|------------|
| `changepoint_prior_scale` | Регуляризация резких изменений тренда |
| `seasonality_prior_scale` | Регуляризация сезонности |
| `holidays_prior_scale` | Регуляризация влияния праздников |
| `seasonality_mode` | `'additive'` или `'multiplicative'` |
| `changepoint_range` | Доля обучающего периода, где возможны изменения тренда |
| `interval_width` | Ширина доверительного интервала |


Модель **Prophet** требует строго определённый формат входных данных — это **таблица (DataFrame)** с двумя обязательными столбцами:

---

**Обязательные столбцы:**

| Название | Тип | Описание |
|----------|-----|----------|
| `ds`     | `datetime` (или строка, распознаваемая как дата) | Метка времени наблюдения |
| `y`      | `float` или `int` | Значение временного ряда (целевой признак) |


In [ ]:
solar_dy_day = pd.DataFrame(solar_dy_day)
solar_dy_day.columns = ["y"]
solar_dy_day["ds"] = solar_dy_day.index

In [ ]:
solar_dy_day.head()

Разделим данные на обучающую и тестовую выборки

In [ ]:
cutoff_date = '2023-01-01'

solar_dy_day_train = solar_dy_day[solar_dy_day["ds"] < cutoff_date].copy()
solar_dy_day_test = solar_dy_day[solar_dy_day["ds"] >= cutoff_date].copy()

print(f"Train: {solar_dy_day_train.shape[0]} записей")
print(f"Test: {solar_dy_day_test.shape[0]} записей")

#### Проверка train/test (дневной solar)

Перед обучением проверяем:
- сортировку и уникальность ds;
- отсутствие пропусков в y;
- корректность размера train/test.

Эта ячейка снижает риск тихих ошибок в метриках и визуализациях.


In [ ]:
assert solar_dy_day_train['ds'].is_monotonic_increasing
assert solar_dy_day_test['ds'].is_monotonic_increasing
assert solar_dy_day_train['ds'].is_unique
assert solar_dy_day_test['ds'].is_unique
assert solar_dy_day_train['y'].notna().all()
assert solar_dy_day_test['y'].notna().all()

print('Проверка пройдена: daily solar train/test корректны.')


Оценка модели через baseline нужна, чтобы понять, есть ли реальный прирост качества.

В этом ноутбуке используем две базовые стратегии:

1. baseline_mean — постоянный прогноз, равный среднему на train.
2. baseline_month — сезонный baseline по месяцу (январь/февраль/...)
   на основе средних значений train.

Почему это важно:
- baseline_mean показывает минимально разумный уровень.
- baseline_month учитывает выраженную годовую/сезонную структуру ряда.


In [ ]:
# Baseline 1: простое среднее по train
# Делаем копию test-таблицы, чтобы не менять исходный DataFrame по ссылке.
solar_dy_day_test = solar_dy_day_test.copy()
# Для baseline_mean каждой дате в тесте назначаем одно и то же значение:
# среднюю генерацию на обучающем интервале.
solar_dy_day_test['baseline_mean'] = solar_dy_day_train['y'].mean()

# Baseline 2: профиль по месяцу (1=январь, ..., 12=декабрь)
# Считаем среднюю генерацию отдельно для каждого месяца на train.
month_profile = solar_dy_day_train.groupby(solar_dy_day_train['ds'].dt.month)['y'].mean()
# Для каждой даты теста берём номер месяца и подставляем соответствующее
# историческое среднее из month_profile.
solar_dy_day_test['baseline_month'] = solar_dy_day_test['ds'].dt.month.map(month_profile)
# Если в тесте встретился месяц, которого не было в train (или значение не найдено),
# используем безопасный fallback: baseline_mean.
solar_dy_day_test['baseline_month'] = solar_dy_day_test['baseline_month'].fillna(solar_dy_day_test['baseline_mean'])

# Оставляем прежнее имя переменной для совместимости с остальными ячейками.
solar_dy_day_baseline_pred = solar_dy_day_test['baseline_mean'].values
solar_dy_day_baseline_month_pred = solar_dy_day_test['baseline_month'].values

print('Готовы baseline-предсказания: baseline_mean и baseline_month')


#### Визуальная проверка baseline (solar daily)

На одном графике показываем:
- обучающую часть (`train`),
- тестовую часть (`test`),
- два baseline-прогноза на тесте (`baseline_mean`, `baseline_month`).

График нужен для быстрой sanity-проверки: видно, насколько baseline повторяет форму тестового участка
и где именно он систематически недооценивает/переоценивает генерацию.


In [ ]:
plt.figure(figsize=(20, 6))

plt.plot(
    solar_dy_day_train['ds'],
    solar_dy_day_train['y'],
    color='tab:blue',
    label='Train',
)

plt.plot(
    solar_dy_day_test['ds'],
    solar_dy_day_test['y'],
    color='black',
    label='Test',
)

plt.plot(
    solar_dy_day_test['ds'],
    solar_dy_day_test['baseline_mean'],
    color='tab:orange',
    linewidth=2,
    label='Baseline mean',
)

plt.plot(
    solar_dy_day_test['ds'],
    solar_dy_day_test['baseline_month'],
    color='tab:green',
    linewidth=2,
    label='Baseline month',
)

plt.axvline(pd.to_datetime(cutoff_date), color='red', linestyle='--', alpha=0.8, label='Train/Test split')
plt.title('Solar daily: train, test, baseline_mean, baseline_month')
plt.xlabel('Date')
plt.ylabel('Generation')
plt.grid(alpha=0.25)
plt.legend()
plt.show()


Настройка и обучение модели предсказания

In [ ]:
from prophet import Prophet

model_param ={
    "daily_seasonality": False, # ежедневная сезонность
    "weekly_seasonality":False, # недельная сезонность
    "yearly_seasonality":True, # годовая сезонность
    # "interval_width": 0.8, # ширина доверительного интервала
    # "seasonality_mode": "multiplicative", # сезонность будет мультипликативной
    # "changepoint_prior_scale" : 0.05 # управляет гибкостью модели в определении изменений тренда
}

model = Prophet(**model_param)

model.fit(solar_dy_day_train);

Получаем прогноз

In [ ]:
future= model.make_future_dataframe(
    periods=solar_dy_day_test.shape[0], # количество будущих временных шагов для прогноза
    freq='d' # частота временного ряда
)
forecast= model.predict(future)

# Выравниваем прогноз с тестом по ds, чтобы исключить ошибки из-за срезов по хвосту
forecast_daily_eval = (
    solar_dy_day_test[['ds', 'y', 'baseline_mean', 'baseline_month']]
    .merge(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds', how='left')
    .sort_values('ds')
    .reset_index(drop=True)
)

assert forecast_daily_eval['yhat'].notna().all(), 'Не все точки теста получили прогноз Prophet'


**forecast** содержит как прогнозы (`yhat`), так и подробную информацию о каждом компоненте модели.

---

**Основные колонки в `forecast`:**

| Колонка         | Описание |
|-----------------|----------|
| `ds`            | Дата/время точки (datetime) — вход |
| `yhat`          | Прогноз модели (сумма всех компонент) |
| `yhat_lower`    | Нижняя граница доверительного интервала |
| `yhat_upper`    | Верхняя граница доверительного интервала |

---

**Компоненты прогноза:**

| Колонка               | Что означает |
|------------------------|---------------|
| `trend`                | Компонент тренда |
| `seasonal`             | Общая сезонность (сумма всех сезонностей) |
| `seasonal_[тип]`       | Сезонность по типу (например, `seasonal_yearly`, `seasonal_weekly`, `seasonal_daily`, если они включены) |
| `holidays`             | Эффект праздников |
| `additive_terms`       | Сумма `seasonal + holidays` (если используется аддитивная модель) |
| `multiplicative_terms` | Мультипликативные эффекты (если включено `seasonality_mode='multiplicative'`) |


In [ ]:
forecast.head()

### Анализ качества модели

In [ ]:
fig = model.plot(
    forecast, xlabel='Datetime(gmt)',
    ylabel=r'Солнечная электроэнергия',
    figsize=(20, 5), uncertainty=True)
plt.title('Генерация солнечной электроэнергии по дням')

ax = fig.gca()

# Добавим тестовые значения
ax.plot(
    solar_dy_day_test['ds'], solar_dy_day_test['y'], 'r.',
    label='Тестовые данные', markersize=4,
    zorder=1)

plt.show()

```plot_components``` построит графики всех доступных составляющих

In [ ]:
fig = model.plot_components(forecast,uncertainty=True,figsize=(20, 10))
plt.show()

Выполним расчет наиболее распространённых метрик

Обозначения:

- $ y_t $ — истинное значение в момент времени $ t $  
- $ \hat{y}_t $ — предсказанное значение  
- $ n $ — общее количество прогнозируемых точек

**MAE (Mean Absolute Error)**

Средняя абсолютная ошибка:

$$ \text{MAE} = \frac{1}{n} \sum_{t=1}^{n} |y_t - \hat{y}_t| $$

In [ ]:
from sklearn.metrics import mean_absolute_error

mae_prophet = mean_absolute_error(
    forecast_daily_eval['y'],
    forecast_daily_eval['yhat']
)
mae_baseline_mean = mean_absolute_error(
    forecast_daily_eval['y'],
    forecast_daily_eval['baseline_mean']
)
mae_baseline_month = mean_absolute_error(
    forecast_daily_eval['y'],
    forecast_daily_eval['baseline_month']
)

print(f"MAE Prophet: {mae_prophet:.2f}")
print(f"MAE baseline_mean: {mae_baseline_mean:.2f}")
print(f"MAE baseline_month: {mae_baseline_month:.2f}")


**RMSE (Root Mean Squared Error)**

Корень из средней квадратичной ошибки:

$$ \text{RMSE} = \sqrt{\frac{1}{n} \sum_{t=1}^{n} (y_t - \hat{y}_t)^2} $$

In [ ]:
from sklearn.metrics import mean_squared_error

rmse_prophet = np.sqrt(mean_squared_error(
    forecast_daily_eval['y'],
    forecast_daily_eval['yhat']
))
rmse_baseline_mean = np.sqrt(mean_squared_error(
    forecast_daily_eval['y'],
    forecast_daily_eval['baseline_mean']
))
rmse_baseline_month = np.sqrt(mean_squared_error(
    forecast_daily_eval['y'],
    forecast_daily_eval['baseline_month']
))

print(f"RMSE Prophet: {rmse_prophet:.2f}")
print(f"RMSE baseline_mean: {rmse_baseline_mean:.2f}")
print(f"RMSE baseline_month: {rmse_baseline_month:.2f}")


**MAPE, sMAPE и WAPE: что это и как интерпретировать**

Пусть:
- $y_t$ — фактическое значение,
- $\hat{y}_t$ — прогноз,
- $n$ — число точек.

1. **MAPE** (Mean Absolute Percentage Error)
$$
\text{MAPE} = \frac{100\%}{n} \sum_{t=1}^{n} \left|\frac{y_t-\hat{y}_t}{y_t}\right|
$$

Плюсы: понятная процентная интерпретация.  
Минусы: не определена при $y_t=0$, может сильно искажаться при очень малых $y_t$.

2. **sMAPE** (Symmetric MAPE)
$$
\text{sMAPE} = \frac{100\%}{n} \sum_{t=1}^{n}
\frac{2|y_t-\hat{y}_t|}{|y_t|+|\hat{y}_t|+\varepsilon}
$$

Более устойчива к малым значениям и симметричнее MAPE.

3. **WAPE** (Weighted Absolute Percentage Error)
$$
\text{WAPE} = \frac{\sum_{t=1}^{n}|y_t-\hat{y}_t|}{\sum_{t=1}^{n}|y_t|+\varepsilon}\cdot100\%
$$

Показывает суммарную относительную ошибку и хорошо работает на рядах с нулями.

В ноутбуке считаем все три метрики.  
Для MAPE используем только точки, где $y_t \neq 0$, и отдельно выводим долю исключённых точек.


In [ ]:
eps = 1e-6

def smape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100


def wape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sum(np.abs(y_pred - y_true)) / (np.sum(np.abs(y_true)) + eps) * 100


def mape_nonzero(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = np.abs(y_true) > eps
    excluded_share = (1.0 - mask.mean()) * 100
    if mask.sum() == 0:
        return np.nan, excluded_share
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return mape, excluded_share

smape_prophet = smape(forecast_daily_eval['y'], forecast_daily_eval['yhat'], eps)
smape_baseline_mean = smape(forecast_daily_eval['y'], forecast_daily_eval['baseline_mean'], eps)
smape_baseline_month = smape(forecast_daily_eval['y'], forecast_daily_eval['baseline_month'], eps)

wape_prophet = wape(forecast_daily_eval['y'], forecast_daily_eval['yhat'], eps)
wape_baseline_mean = wape(forecast_daily_eval['y'], forecast_daily_eval['baseline_mean'], eps)
wape_baseline_month = wape(forecast_daily_eval['y'], forecast_daily_eval['baseline_month'], eps)

mape_prophet, excl_prophet = mape_nonzero(forecast_daily_eval['y'], forecast_daily_eval['yhat'], eps)
mape_baseline_mean, excl_mean = mape_nonzero(forecast_daily_eval['y'], forecast_daily_eval['baseline_mean'], eps)
mape_baseline_month, excl_month = mape_nonzero(forecast_daily_eval['y'], forecast_daily_eval['baseline_month'], eps)

print(f"MAPE Prophet (y!=0): {mape_prophet:.2f}%")
print(f"MAPE baseline_mean (y!=0): {mape_baseline_mean:.2f}%")
print(f"MAPE baseline_month (y!=0): {mape_baseline_month:.2f}%")
print(f"Доля исключённых точек для MAPE: Prophet={excl_prophet:.2f}%, baseline_mean={excl_mean:.2f}%, baseline_month={excl_month:.2f}%")
print('---')
print(f"sMAPE Prophet: {smape_prophet:.2f}%")
print(f"sMAPE baseline_mean: {smape_baseline_mean:.2f}%")
print(f"sMAPE baseline_month: {smape_baseline_month:.2f}%")
print('---')
print(f"WAPE Prophet: {wape_prophet:.2f}%")
print(f"WAPE baseline_mean: {wape_baseline_mean:.2f}%")
print(f"WAPE baseline_month: {wape_baseline_month:.2f}%")

metrics_daily = pd.DataFrame([
    {'scenario': 'solar_daily', 'model': 'prophet', 'MAE': mae_prophet, 'RMSE': rmse_prophet, 'MAPE': mape_prophet, 'sMAPE': smape_prophet, 'WAPE': wape_prophet},
    {'scenario': 'solar_daily', 'model': 'baseline_mean', 'MAE': mae_baseline_mean, 'RMSE': rmse_baseline_mean, 'MAPE': mape_baseline_mean, 'sMAPE': smape_baseline_mean, 'WAPE': wape_baseline_mean},
    {'scenario': 'solar_daily', 'model': 'baseline_month', 'MAE': mae_baseline_month, 'RMSE': rmse_baseline_month, 'MAPE': mape_baseline_month, 'sMAPE': smape_baseline_month, 'WAPE': wape_baseline_month},
])


С точки зрения средней ошибки модель имеет очень низкое качество. Посмотрим на суммартные ошибки

**Покрытие (Coverage )**

Доля истинных значений, попавших в доверительный интервал прогноза

Если есть предсказания нижней и верхней границы интервала: $ \hat{y}_t^{\text{lower}}, \hat{y}_t^{\text{upper}} $, тогда:

$$ \text{Coverage} = \frac{1}{n} \sum_{t=1}^{n} \mathbb{1} \left[ \hat{y}_t^{\text{lower}} \leq y_t \leq \hat{y}_t^{\text{upper}} \right] $$

где $ \mathbb{1}[\cdot] $ — индикаторная функция (1, если условие выполнено, 0 иначе).

In [ ]:
above_lower = forecast_daily_eval['y'] >= forecast_daily_eval['yhat_lower']
below_upper = forecast_daily_eval['y'] <= forecast_daily_eval['yhat_upper']
in_interval = above_lower & below_upper

coverage = in_interval.mean()
print(f"Покрытие: {coverage*100:.2f}%")

metrics_daily['Coverage'] = [coverage, np.nan, np.nan]


### Опциональная валидация по rolling-window (time-series CV)

Одна точка разреза train/test полезна, но может давать нестабильную оценку.

Для более надежной диагностики можно включить RUN_CV = True и получить
метрики на нескольких исторических окнах.

Важно:
- расчет может быть долгим;
- параметры initial/period/horizon стоит подбирать под бизнес-горизонт прогноза.


In [ ]:
RUN_CV = True

if RUN_CV:
    from prophet.diagnostics import cross_validation, performance_metrics

    # Явно считаем ожидаемое количество CV-окон (cutoff) по формуле Prophet
    initial = '730 days'
    period = '90 days'
    horizon = '180 days'

    cv_df = cross_validation(
        model,
        initial=initial,
        period=period,
        horizon=horizon,
        parallel=None,
    )

    actual_cutoffs = cv_df['cutoff'].nunique()
    print(f'Фактическое число cutoff: {actual_cutoffs}')
    print(f'Строк в cv_df: {len(cv_df)}')

    cv_metrics = performance_metrics(cv_df, rolling_window=1)
    display(cv_metrics[['horizon', 'mae', 'rmse', 'mape', 'coverage']].head())
else:
    print('CV выключен (RUN_CV=False). Для запуска переключите флаг на True.')


### Предсказание производства солнечной электроэнергии с учетом дневной сезонности

In [ ]:
solar = pd.DataFrame(solar)
solar.columns = ["y"]
solar["ds"] = solar.index

In [ ]:
solar.head()

Разделим данные на обучающую и тестовую выборки

In [ ]:
cutoff_date = '2023-01-01'

solar_train = solar[solar["ds"] < cutoff_date].copy()
solar_test = solar[solar["ds"] >= cutoff_date].copy()

print(f"Train: {solar_train.shape[0]} записей")
print(f"Test: {solar_test.shape[0]} записей")

#### Проверка train/test (hourly solar)

Дополнительно выводим долю нулей в тесте, чтобы заранее понимать,
почему процентные метрики типа MAPE могут быть нестабильны.


In [ ]:
assert solar_train['ds'].is_monotonic_increasing
assert solar_test['ds'].is_monotonic_increasing
assert solar_train['ds'].is_unique
assert solar_test['ds'].is_unique
assert solar_train['y'].notna().all()
assert solar_test['y'].notna().all()

zero_share = (solar_test['y'] == 0).mean() * 100
print(f'Проверка пройдена: hourly solar корректен. Доля нулей в test: {zero_share:.2f}%')


Базовые модели:

1. baseline_mean — среднее train.
2. baseline_monthhour — среднее по паре (месяц, час) на train.

Для этого ряда month+hour обычно лучше отражает сезонность, чем dayofweek+hour.


In [ ]:
# Baseline 1: среднее значение
# Работаем с копией test-набора, чтобы не менять исходные данные выше по ноутбуку.
solar_test = solar_test.copy()
# Для baseline_mean всем точкам теста задаётся одна константа:
# среднее значение y на train.
solar_test['baseline_mean'] = solar_train['y'].mean()

# Baseline 2: профиль по (месяц, час)
# На train строим таблицу средних для каждой пары (month, hour).
monthhour_profile = solar_train.groupby([
    solar_train['ds'].dt.month,
    solar_train['ds'].dt.hour,
])['y'].mean()

# Дополнительный fallback: среднее только по часу суток.
hour_profile = solar_train.groupby(solar_train['ds'].dt.hour)['y'].mean()

# Для каждой строки теста пытаемся взять значение из monthhour_profile
# по ключу (месяц, час). Если пары нет, временно ставим NaN.
solar_test['baseline_monthhour'] = [
    monthhour_profile.get((m, hour), np.nan)
    for m, hour in zip(solar_test['ds'].dt.month, solar_test['ds'].dt.hour)
]
# 1-й fallback: если пары (месяц, час) не нашлось, берём среднее по этому часу.
solar_test['baseline_monthhour'] = solar_test['baseline_monthhour'].fillna(
    solar_test['ds'].dt.hour.map(hour_profile)
)
# 2-й fallback: если и час не найден, подставляем baseline_mean.
solar_test['baseline_monthhour'] = solar_test['baseline_monthhour'].fillna(solar_test['baseline_mean'])

# Массивы оставляем в прежних переменных, чтобы не ломать следующие ячейки.
solar_baseline_pred = solar_test['baseline_mean'].values
solar_baseline_monthhour_pred = solar_test['baseline_monthhour'].values

print('Готовы baseline-предсказания: baseline_mean и baseline_monthhour')


#### Визуальная проверка baseline (solar hourly)

Здесь сравниваем `train`, `test` и два baseline на почасовом ряду:
- `baseline_mean`,
- `baseline_monthhour`.

Для плотного hourly-ряда используем полупрозрачность, чтобы график оставался читаемым.


In [ ]:
plt.figure(figsize=(20, 6))

plt.plot(
    solar_train['ds'],
    solar_train['y'],
    color='tab:blue',
    alpha=0.5,
    label='Train',
    zorder=1,
)

plt.plot(
    solar_test['ds'],
    solar_test['y'],
    color='black',
    alpha=0.7,
    label='Test',
    zorder=2,
)

# Baseline 1 лучше рисовать отдельной горизонтальной линией,
# чтобы она не терялась на плотном графике.
baseline_mean_value = solar_test['baseline_mean'].iloc[0]
plt.hlines(
    y=baseline_mean_value,
    xmin=solar_test['ds'].min(),
    xmax=solar_test['ds'].max(),
    color='tab:orange',
    linewidth=2.5,
    linestyle='--',
    label=f'Baseline mean ({baseline_mean_value:.1f})',
    zorder=5,
)

plt.plot(
    solar_test['ds'],
    solar_test['baseline_monthhour'],
    color='tab:green',
    linewidth=1.5,
    label='Baseline monthhour',
    zorder=3,
)

plt.axvline(pd.to_datetime(cutoff_date), color='red', linestyle='--', alpha=0.8, label='Train/Test split')
plt.title('Solar hourly: train, test, baseline_mean, baseline_monthhour')
plt.xlabel('DateTime')
plt.ylabel('Generation')
plt.grid(alpha=0.25)
plt.legend()
plt.show()


Настройка и обучение модели предсказания

In [ ]:
from prophet import Prophet

model_param ={
    "daily_seasonality": True, # ежедневная сезонность
    "weekly_seasonality":True, # недельная сезонность
    "yearly_seasonality":True, # годовая сезонность
    # "seasonality_mode": "multiplicative", # сезонность будет мультипликативной
    # "changepoint_prior_scale" : 0.05 # управляет гибкостью модели в определении изменений тренда
}

model = Prophet(**model_param)

model.fit(solar_train);

Получаем прогноз

In [ ]:
future= model.make_future_dataframe(
    periods=solar_test.shape[0], # количество будущих временных шагов для прогноза
    freq='h' # частота временного ряда
)
forecast= model.predict(future)

forecast_solar_hourly_eval = (
    solar_test[['ds', 'y', 'baseline_mean', 'baseline_monthhour']]
    .merge(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds', how='left')
    .sort_values('ds')
    .reset_index(drop=True)
)

missing_pred = forecast_solar_hourly_eval['yhat'].isna().sum()
if missing_pred > 0:
    print(f'Внимание: отсутствует прогноз для {missing_pred} точек hourly solar. Они будут исключены из оценки.')
forecast_solar_hourly_eval = forecast_solar_hourly_eval.dropna(subset=['yhat', 'yhat_lower', 'yhat_upper']).reset_index(drop=True)


### Анализ качества модели

In [ ]:
fig = model.plot(
    forecast, xlabel='Datetime(gmt)', ylabel=r'Солнечная электроэнергия', figsize=(20, 5), uncertainty=True)
plt.title('Почасовая генерация солнечной электроэнергии')

ax = fig.gca()

# Добавим тестовые значения
ax.plot(
    solar_test['ds'], solar_test['y'], 'r.',
    label='Тестовые данные', markersize=4,
    zorder=1)

plt.show()

In [ ]:
solar_test['y'].plot(
          style='.',
          figsize=(20, 5),
          ms=10,
          color="black",
          title='Солнечная электроэнергия');

plt.plot(
    forecast_solar_hourly_eval['ds'],
    forecast_solar_hourly_eval['yhat'].clip(lower=0));

plt.fill_between(
    forecast_solar_hourly_eval['ds'],
    forecast_solar_hourly_eval['yhat_lower'].clip(lower=0),
    forecast_solar_hourly_eval['yhat_upper'].clip(lower=0),
    color='skyblue',
    alpha=0.4, label='Доверительный интервал (80%)');

plt.xlim(pd.to_datetime([solar_test.index[-240], solar_test.index[-1]]));
plt.legend();


In [ ]:
fig = model.plot_components(forecast,uncertainty=True,figsize=(20, 10))
plt.show()

**MAE (Mean Absolute Error)**

Средняя абсолютная ошибка:

In [ ]:
from sklearn.metrics import mean_absolute_error

mae_prophet = mean_absolute_error(
    forecast_solar_hourly_eval['y'],
    forecast_solar_hourly_eval['yhat'].clip(lower=0)
)
mae_baseline_mean = mean_absolute_error(
    forecast_solar_hourly_eval['y'],
    forecast_solar_hourly_eval['baseline_mean']
)
mae_baseline_monthhour = mean_absolute_error(
    forecast_solar_hourly_eval['y'],
    forecast_solar_hourly_eval['baseline_monthhour']
)

print(f"MAE Prophet: {mae_prophet:.2f}")
print(f"MAE baseline_mean: {mae_baseline_mean:.2f}")
print(f"MAE baseline_monthhour: {mae_baseline_monthhour:.2f}")


**RMSE**

In [ ]:
from sklearn.metrics import mean_squared_error

rmse_prophet = np.sqrt(mean_squared_error(
    forecast_solar_hourly_eval['y'],
    forecast_solar_hourly_eval['yhat'].clip(lower=0)
))
rmse_baseline_mean = np.sqrt(mean_squared_error(
    forecast_solar_hourly_eval['y'],
    forecast_solar_hourly_eval['baseline_mean']
))
rmse_baseline_monthhour = np.sqrt(mean_squared_error(
    forecast_solar_hourly_eval['y'],
    forecast_solar_hourly_eval['baseline_monthhour']
))

print(f"RMSE Prophet: {rmse_prophet:.2f}")
print(f"RMSE baseline_mean: {rmse_baseline_mean:.2f}")
print(f"RMSE baseline_monthhour: {rmse_baseline_monthhour:.2f}")


**MAPE, sMAPE, WAPE**

In [ ]:
eps = 1e-6

def mape_nonzero(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = np.abs(y_true) > eps
    excluded_share = (1.0 - mask.mean()) * 100
    if mask.sum() == 0:
        return np.nan, excluded_share
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return mape, excluded_share

smape_prophet = np.mean(
    2.0 * np.abs(forecast_solar_hourly_eval['yhat'].clip(lower=0) - forecast_solar_hourly_eval['y']) /
    (np.abs(forecast_solar_hourly_eval['y']) + np.abs(forecast_solar_hourly_eval['yhat'].clip(lower=0)) + eps)
) * 100
smape_baseline_mean = np.mean(
    2.0 * np.abs(forecast_solar_hourly_eval['baseline_mean'] - forecast_solar_hourly_eval['y']) /
    (np.abs(forecast_solar_hourly_eval['y']) + np.abs(forecast_solar_hourly_eval['baseline_mean']) + eps)
) * 100
smape_baseline_monthhour = np.mean(
    2.0 * np.abs(forecast_solar_hourly_eval['baseline_monthhour'] - forecast_solar_hourly_eval['y']) /
    (np.abs(forecast_solar_hourly_eval['y']) + np.abs(forecast_solar_hourly_eval['baseline_monthhour']) + eps)
) * 100

wape_prophet = (
    np.abs(forecast_solar_hourly_eval['yhat'].clip(lower=0) - forecast_solar_hourly_eval['y']).sum() /
    (np.abs(forecast_solar_hourly_eval['y']).sum() + eps)
) * 100
wape_baseline_mean = (
    np.abs(forecast_solar_hourly_eval['baseline_mean'] - forecast_solar_hourly_eval['y']).sum() /
    (np.abs(forecast_solar_hourly_eval['y']).sum() + eps)
) * 100
wape_baseline_monthhour = (
    np.abs(forecast_solar_hourly_eval['baseline_monthhour'] - forecast_solar_hourly_eval['y']).sum() /
    (np.abs(forecast_solar_hourly_eval['y']).sum() + eps)
) * 100

mape_prophet, excl_prophet = mape_nonzero(forecast_solar_hourly_eval['y'], forecast_solar_hourly_eval['yhat'].clip(lower=0), eps)
mape_baseline_mean, excl_mean = mape_nonzero(forecast_solar_hourly_eval['y'], forecast_solar_hourly_eval['baseline_mean'], eps)
mape_baseline_monthhour, excl_monthhour = mape_nonzero(forecast_solar_hourly_eval['y'], forecast_solar_hourly_eval['baseline_monthhour'], eps)

print(f"MAPE Prophet (y!=0): {mape_prophet:.2f}%")
print(f"MAPE baseline_mean (y!=0): {mape_baseline_mean:.2f}%")
print(f"MAPE baseline_monthhour (y!=0): {mape_baseline_monthhour:.2f}%")
print(f"Доля исключённых точек для MAPE: Prophet={excl_prophet:.2f}%, baseline_mean={excl_mean:.2f}%, baseline_monthhour={excl_monthhour:.2f}%")
print('---')
print(f"sMAPE Prophet: {smape_prophet:.2f}%")
print(f"sMAPE baseline_mean: {smape_baseline_mean:.2f}%")
print(f"sMAPE baseline_monthhour: {smape_baseline_monthhour:.2f}%")
print('---')
print(f"WAPE Prophet: {wape_prophet:.2f}%")
print(f"WAPE baseline_mean: {wape_baseline_mean:.2f}%")
print(f"WAPE baseline_monthhour: {wape_baseline_monthhour:.2f}%")

metrics_solar_hourly = pd.DataFrame([
    {'scenario': 'solar_hourly', 'model': 'prophet', 'MAE': mae_prophet, 'RMSE': rmse_prophet, 'MAPE': mape_prophet, 'sMAPE': smape_prophet, 'WAPE': wape_prophet},
    {'scenario': 'solar_hourly', 'model': 'baseline_mean', 'MAE': mae_baseline_mean, 'RMSE': rmse_baseline_mean, 'MAPE': mape_baseline_mean, 'sMAPE': smape_baseline_mean, 'WAPE': wape_baseline_mean},
    {'scenario': 'solar_hourly', 'model': 'baseline_monthhour', 'MAE': mae_baseline_monthhour, 'RMSE': rmse_baseline_monthhour, 'MAPE': mape_baseline_monthhour, 'sMAPE': smape_baseline_monthhour, 'WAPE': wape_baseline_monthhour},
])


**Покрытие доверительного интервала (Coverage)**


In [ ]:
above_lower = forecast_solar_hourly_eval['y'] >= forecast_solar_hourly_eval['yhat_lower']
below_upper = forecast_solar_hourly_eval['y'] <= forecast_solar_hourly_eval['yhat_upper']
in_interval = above_lower & below_upper

coverage = in_interval.mean()
print(f"Покрытие: {coverage*100:.2f}%")

metrics_solar_hourly['Coverage'] = [coverage, np.nan, np.nan]


## Производство ветряной электроэнергии
### Разложение временного ряда

In [ ]:
_ = seasonal_decompose_plotter(wind, period=365*24, title='Wind Power Generation Seasonal Decompose', figsize=(20, 12))

In [ ]:
fig = plt.figure(figsize=(20, 3));
plt.plot(wind.index, _.seasonal, color='C2');
plt.xlim(pd.to_datetime(['2022-12-01', '2022-12-14']));

### Предсказание производства ветряной электроэнергии

In [ ]:
wind = pd.DataFrame(wind)
wind.columns = ["y"]
wind["ds"] = wind.index

Разделим данные на обучающую и тестовую выборки

In [ ]:
cutoff_date = '2023-01-01'

wind_train = wind[wind["ds"] < cutoff_date].copy()
wind_test = wind[wind["ds"] >= cutoff_date].copy()

print(f"Train: {wind_train.shape[0]} записей")
print(f"Test: {wind_test.shape[0]} записей")

#### Проверка train/test (hourly wind)

Проверяем структуру данных перед обучением и вычисляем долю нулевых значений в тесте.


In [ ]:
assert wind_train['ds'].is_monotonic_increasing
assert wind_test['ds'].is_monotonic_increasing
assert wind_train['ds'].is_unique
assert wind_test['ds'].is_unique
assert wind_train['y'].notna().all()
assert wind_test['y'].notna().all()

zero_share = (wind_test['y'] == 0).mean() * 100
print(f'Проверка пройдена: hourly wind корректен. Доля нулей в test: {zero_share:.2f}%')


Базовые модели:

1. baseline_mean — среднее train.
2. baseline_monthhour — среднее по паре (месяц, час) на train.

Для этого ряда month+hour обычно дает более устойчивый baseline.


In [ ]:
# Baseline 1: среднее значение
# Работаем на копии test, чтобы не менять объект в других частях ноутбука.
wind_test = wind_test.copy()
# baseline_mean: постоянный прогноз, равный среднему y на train.
wind_test['baseline_mean'] = wind_train['y'].mean()

# Baseline 2: профиль по (месяц, час)
# Строим средние значения на train для каждой комбинации (month, hour).
monthhour_profile = wind_train.groupby([
    wind_train['ds'].dt.month,
    wind_train['ds'].dt.hour,
])['y'].mean()

# Резервный профиль по одному признаку hour.
hour_profile = wind_train.groupby(wind_train['ds'].dt.hour)['y'].mean()

# Основной расчёт baseline 2: lookup по ключу (месяц, час) для каждой точки теста.
wind_test['baseline_monthhour'] = [
    monthhour_profile.get((m, hour), np.nan)
    for m, hour in zip(wind_test['ds'].dt.month, wind_test['ds'].dt.hour)
]
# Если комбинации нет в train, используем среднее по часу суток.
wind_test['baseline_monthhour'] = wind_test['baseline_monthhour'].fillna(
    wind_test['ds'].dt.hour.map(hour_profile)
)
# Если и это не помогло, окончательный fallback — baseline_mean.
wind_test['baseline_monthhour'] = wind_test['baseline_monthhour'].fillna(wind_test['baseline_mean'])

# Сохраняем прогнозы в отдельных массивах для последующих метрик и графиков.
wind_baseline_pred = wind_test['baseline_mean'].values
wind_baseline_monthhour_pred = wind_test['baseline_monthhour'].values

print('Готовы baseline-предсказания: baseline_mean и baseline_monthhour')


#### Визуальная проверка baseline (wind hourly)

На графике выводим:
- `train`,
- `test`,
- `baseline_mean`,
- `baseline_monthhour`.

Это позволяет визуально проверить, насколько сезонный baseline по `(month, hour)`
лучше улавливает структуру тестовой выборки, чем постоянный baseline.


In [ ]:
plt.figure(figsize=(20, 6))

plt.plot(
    wind_train['ds'],
    wind_train['y'],
    color='tab:blue',
    alpha=0.5,
    label='Train',
)

plt.plot(
    wind_test['ds'],
    wind_test['y'],
    color='black',
    alpha=0.7,
    label='Test',
)

plt.plot(
    wind_test['ds'],
    wind_test['baseline_mean'],
    color='tab:orange',
    linewidth=1.5,
    label='Baseline mean',
)

plt.plot(
    wind_test['ds'],
    wind_test['baseline_monthhour'],
    color='tab:green',
    linewidth=1.5,
    label='Baseline monthhour',
)

plt.axvline(pd.to_datetime(cutoff_date), color='red', linestyle='--', alpha=0.8, label='Train/Test split')
plt.title('Wind hourly: train, test, baseline_mean, baseline_monthhour')
plt.xlabel('DateTime')
plt.ylabel('Generation')
plt.grid(alpha=0.25)
plt.legend()
plt.show()


Настройка и обучение модели предсказания

In [ ]:
model_param ={
    "daily_seasonality": False,
    "weekly_seasonality":False,
    "yearly_seasonality":True,
    # "seasonality_mode": "multiplicative",
    # "changepoint_prior_scale" : 0.5
}

model = Prophet(**model_param)
model.fit(wind_train);

Получаем прогноз

In [ ]:
# Create future dataframe
future= model.make_future_dataframe(periods=wind_test.shape[0] ,freq='h')
forecast= model.predict(future)

forecast_wind_hourly_eval = (
    wind_test[['ds', 'y', 'baseline_mean', 'baseline_monthhour']]
    .merge(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds', how='left')
    .sort_values('ds')
    .reset_index(drop=True)
)

missing_pred = forecast_wind_hourly_eval['yhat'].isna().sum()
if missing_pred > 0:
    print(f'Внимание: отсутствует прогноз для {missing_pred} точек hourly wind. Они будут исключены из оценки.')
forecast_wind_hourly_eval = forecast_wind_hourly_eval.dropna(subset=['yhat', 'yhat_lower', 'yhat_upper']).reset_index(drop=True)


### Анализ качества модели

In [ ]:
fig = model.plot(
    forecast, xlabel='Datetime(gmt)',
    ylabel=r'Ветряная электроэнергия', figsize=(20, 5), uncertainty=True)

ax = fig.gca()

# Увеличиваем толщину линии прогноза (она всегда синяя по умолчанию):
for line in ax.get_lines():
    print(line.get_label())
    if line.get_label() == 'Forecast':
        line.set_linewidth(5)

# Добавим тестовые значения
ax.plot(
    wind_test['ds'], wind_test['y'], 'r.',
    label='Тестовые данные', markersize=4,
    zorder=1)

plt.title('Почасовая генерация ветряной электроэнергии')
plt.show()

In [ ]:
wind_test['y'].plot(
          style='.',
          figsize=(20, 5),
          ms=10,
          color="black",
          title='Ветряная электроэнергия');

plt.plot(
    forecast_wind_hourly_eval['ds'],
    forecast_wind_hourly_eval['yhat'].clip(lower=0));

plt.fill_between(
    forecast_wind_hourly_eval['ds'],
    forecast_wind_hourly_eval['yhat_lower'].clip(lower=0),
    forecast_wind_hourly_eval['yhat_upper'].clip(lower=0),
    color='skyblue',
    alpha=0.4, label='Доверительный интервал (80%)');

plt.xlim(pd.to_datetime([wind_test.index[0], wind_test.index[-1]]));
plt.legend();


In [ ]:
fig = model.plot_components(forecast,uncertainty=True,figsize=(20, 10))
plt.show()

**MAE (Mean Absolute Error)**

Средняя абсолютная ошибка:

In [ ]:
from sklearn.metrics import mean_absolute_error

mae_prophet = mean_absolute_error(
    forecast_wind_hourly_eval['y'],
    forecast_wind_hourly_eval['yhat'].clip(lower=0)
)
mae_baseline_mean = mean_absolute_error(
    forecast_wind_hourly_eval['y'],
    forecast_wind_hourly_eval['baseline_mean']
)
mae_baseline_monthhour = mean_absolute_error(
    forecast_wind_hourly_eval['y'],
    forecast_wind_hourly_eval['baseline_monthhour']
)

print(f"MAE Prophet: {mae_prophet:.2f}")
print(f"MAE baseline_mean: {mae_baseline_mean:.2f}")
print(f"MAE baseline_monthhour: {mae_baseline_monthhour:.2f}")


**RMSE (Root Mean Squared Error)**

Корень из средней квадратичной ошибки:

In [ ]:
from sklearn.metrics import mean_squared_error

rmse_prophet = np.sqrt(mean_squared_error(
    forecast_wind_hourly_eval['y'],
    forecast_wind_hourly_eval['yhat'].clip(lower=0)
))
rmse_baseline_mean = np.sqrt(mean_squared_error(
    forecast_wind_hourly_eval['y'],
    forecast_wind_hourly_eval['baseline_mean']
))
rmse_baseline_monthhour = np.sqrt(mean_squared_error(
    forecast_wind_hourly_eval['y'],
    forecast_wind_hourly_eval['baseline_monthhour']
))

print(f"RMSE Prophet: {rmse_prophet:.2f}")
print(f"RMSE baseline_mean: {rmse_baseline_mean:.2f}")
print(f"RMSE baseline_monthhour: {rmse_baseline_monthhour:.2f}")


**MAPE, sMAPE и WAPE: что это и как интерпретировать**

Пусть:
- $y_t$ — фактическое значение,
- $\hat{y}_t$ — прогноз,
- $n$ — число точек.

1. **MAPE** (Mean Absolute Percentage Error)
$$
\text{MAPE} = \frac{100\%}{n} \sum_{t=1}^{n} \left|\frac{y_t-\hat{y}_t}{y_t}\right|
$$

Плюсы: понятная процентная интерпретация.  
Минусы: не определена при $y_t=0$, может сильно искажаться при очень малых $y_t$.

2. **sMAPE** (Symmetric MAPE)
$$
\text{sMAPE} = \frac{100\%}{n} \sum_{t=1}^{n}
\frac{2|y_t-\hat{y}_t|}{|y_t|+|\hat{y}_t|+\varepsilon}
$$

Более устойчива к малым значениям и симметричнее MAPE.

3. **WAPE** (Weighted Absolute Percentage Error)
$$
\text{WAPE} = \frac{\sum_{t=1}^{n}|y_t-\hat{y}_t|}{\sum_{t=1}^{n}|y_t|+\varepsilon}\cdot100\%
$$

Показывает суммарную относительную ошибку и хорошо работает на рядах с нулями.

В ноутбуке считаем все три метрики.  
Для MAPE используем только точки, где $y_t \neq 0$, и отдельно выводим долю исключённых точек.


In [ ]:
eps = 1e-6

def mape_nonzero(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = np.abs(y_true) > eps
    excluded_share = (1.0 - mask.mean()) * 100
    if mask.sum() == 0:
        return np.nan, excluded_share
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return mape, excluded_share

smape_prophet = np.mean(
    2.0 * np.abs(forecast_wind_hourly_eval['yhat'].clip(lower=0) - forecast_wind_hourly_eval['y']) /
    (np.abs(forecast_wind_hourly_eval['y']) + np.abs(forecast_wind_hourly_eval['yhat'].clip(lower=0)) + eps)
) * 100
smape_baseline_mean = np.mean(
    2.0 * np.abs(forecast_wind_hourly_eval['baseline_mean'] - forecast_wind_hourly_eval['y']) /
    (np.abs(forecast_wind_hourly_eval['y']) + np.abs(forecast_wind_hourly_eval['baseline_mean']) + eps)
) * 100
smape_baseline_monthhour = np.mean(
    2.0 * np.abs(forecast_wind_hourly_eval['baseline_monthhour'] - forecast_wind_hourly_eval['y']) /
    (np.abs(forecast_wind_hourly_eval['y']) + np.abs(forecast_wind_hourly_eval['baseline_monthhour']) + eps)
) * 100

wape_prophet = (
    np.abs(forecast_wind_hourly_eval['yhat'].clip(lower=0) - forecast_wind_hourly_eval['y']).sum() /
    (np.abs(forecast_wind_hourly_eval['y']).sum() + eps)
) * 100
wape_baseline_mean = (
    np.abs(forecast_wind_hourly_eval['baseline_mean'] - forecast_wind_hourly_eval['y']).sum() /
    (np.abs(forecast_wind_hourly_eval['y']).sum() + eps)
) * 100
wape_baseline_monthhour = (
    np.abs(forecast_wind_hourly_eval['baseline_monthhour'] - forecast_wind_hourly_eval['y']).sum() /
    (np.abs(forecast_wind_hourly_eval['y']).sum() + eps)
) * 100

mape_prophet, excl_prophet = mape_nonzero(forecast_wind_hourly_eval['y'], forecast_wind_hourly_eval['yhat'].clip(lower=0), eps)
mape_baseline_mean, excl_mean = mape_nonzero(forecast_wind_hourly_eval['y'], forecast_wind_hourly_eval['baseline_mean'], eps)
mape_baseline_monthhour, excl_monthhour = mape_nonzero(forecast_wind_hourly_eval['y'], forecast_wind_hourly_eval['baseline_monthhour'], eps)

print(f"MAPE Prophet (y!=0): {mape_prophet:.2f}%")
print(f"MAPE baseline_mean (y!=0): {mape_baseline_mean:.2f}%")
print(f"MAPE baseline_monthhour (y!=0): {mape_baseline_monthhour:.2f}%")
print(f"Доля исключённых точек для MAPE: Prophet={excl_prophet:.2f}%, baseline_mean={excl_mean:.2f}%, baseline_monthhour={excl_monthhour:.2f}%")
print('---')
print(f"sMAPE Prophet: {smape_prophet:.2f}%")
print(f"sMAPE baseline_mean: {smape_baseline_mean:.2f}%")
print(f"sMAPE baseline_monthhour: {smape_baseline_monthhour:.2f}%")
print('---')
print(f"WAPE Prophet: {wape_prophet:.2f}%")
print(f"WAPE baseline_mean: {wape_baseline_mean:.2f}%")
print(f"WAPE baseline_monthhour: {wape_baseline_monthhour:.2f}%")

metrics_wind_hourly = pd.DataFrame([
    {'scenario': 'wind_hourly', 'model': 'prophet', 'MAE': mae_prophet, 'RMSE': rmse_prophet, 'MAPE': mape_prophet, 'sMAPE': smape_prophet, 'WAPE': wape_prophet},
    {'scenario': 'wind_hourly', 'model': 'baseline_mean', 'MAE': mae_baseline_mean, 'RMSE': rmse_baseline_mean, 'MAPE': mape_baseline_mean, 'sMAPE': smape_baseline_mean, 'WAPE': wape_baseline_mean},
    {'scenario': 'wind_hourly', 'model': 'baseline_monthhour', 'MAE': mae_baseline_monthhour, 'RMSE': rmse_baseline_monthhour, 'MAPE': mape_baseline_monthhour, 'sMAPE': smape_baseline_monthhour, 'WAPE': wape_baseline_monthhour},
])


**Покрытие доверительного интервала (Coverage)**


In [ ]:
above_lower = forecast_wind_hourly_eval['y'] >= forecast_wind_hourly_eval['yhat_lower']
below_upper = forecast_wind_hourly_eval['y'] <= forecast_wind_hourly_eval['yhat_upper']
in_interval = above_lower & below_upper

coverage = in_interval.mean()
print(f"Покрытие: {coverage*100:.2f}%")

metrics_wind_hourly['Coverage'] = [coverage, np.nan, np.nan]


## Итоговое сравнение моделей

В этой таблице собраны метрики по всем трем сценариям:
- solar_daily
- solar_hourly
- wind_hourly

Колонки:
- MAE, RMSE — абсолютные/квадратичные ошибки;
- sMAPE, WAPE — устойчивые процентные ошибки;
- Coverage — покрытие доверительного интервала Prophet (для baseline не рассчитывается).

Такой сводный вид упрощает выбор модели и проверку, где Prophet действительно дает выигрыш
относительно сильных baseline.


In [ ]:
summary_metrics = pd.concat(
    [metrics_daily, metrics_solar_hourly, metrics_wind_hourly],
    ignore_index=True,
)

summary_metrics = summary_metrics[[
    'scenario', 'model', 'MAE', 'RMSE', 'MAPE', 'sMAPE', 'WAPE', 'Coverage'
]]

summary_metrics = summary_metrics.sort_values(['scenario', 'RMSE']).reset_index(drop=True)
display(summary_metrics)
